# Tutorial 4: Live Waveform & Spectral Streaming

This tutorial demonstrates real-time data access via WebSocket connections
to the EQ gateway.

**What you will learn:**
1. Connect to the spectral WebSocket for live FFT data
2. Reconfigure the spectral stream (channels, mode, FFT size)
3. Connect to the CPOW waveform WebSocket for raw sample data
4. Capture a snapshot of live waveform data to a parquet file

**Prerequisites:**
- A running EQ gateway at `http://localhost:8080`
- The `equser[analysis]` package (`pip install equser[analysis]`)

**Data formats:**
- **Spectral stream**: Arrow IPC binary messages (one FFT window per message; per-window metadata in the Arrow schema)
- **CPOW stream**: Arrow IPC binary messages (~512 rows / 16 ms per message at 32 kHz)

## 1. Setup

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import websocket

%matplotlib inline

from equser.api import GatewayClient, connect_cpow_stream, connect_spectral_stream

GATEWAY = 'http://localhost:8080'
client = GatewayClient(GATEWAY)

In [ ]:
# Discover devices so we can pick a device_id
devices = client.list_devices()
device_id = devices[0]['id'] if devices else 'wave-001'
print(f"Using device: {device_id}")

## 2. Spectral streaming

The spectral WebSocket at `/api/ws/spectral` is a broadcast consumer of the
live CPOW feed (it is **not** per-device). Each binary message is a complete
Arrow IPC stream holding the FFT magnitude spectrum: a shared `frequency_hz`
column plus a `{CHANNEL}_mag_db` column per subscribed channel. Per-window
metadata (`window_start_ts`, `fundamental_hz_{C}`, `thd_pct_{C}`, `pll_locked`,
`fft_size`, ...) travels in the table's `schema.metadata`. Text messages are
JSON gap markers (`{"type": "gap", "skipped_samples": N}`).

**Query parameters:**
- `channels`: channels to analyze from `IA, VA, IB, VB, IC, VC, IN` (default `VA,IA`)
- `mode`: `cycle_aligned` (one window per `cycles` PLL-locked cycles) or `fixed` (one window per `fft_size` samples, ignores PLL lock)
- `cycles`: cycles per window in `cycle_aligned` mode (default 12)
- `fft_size`: window size in `fixed` mode (default 4096, power of 2)
- `freq_min`, `freq_max`: frequency range in Hz (defaults 0–3000)
- `include_phase`: also return `{CHANNEL}_phase_rad` columns

In [ ]:
# Collect a few spectral windows (VA channel, cycle-aligned default)
windows = []
max_windows = 5

print(f"Collecting {max_windows} spectral windows...")
for frame in connect_spectral_stream(channels=['VA'], gateway_url=GATEWAY):
    if isinstance(frame, dict):
        print(f"  Gap: {frame.get('skipped_samples', '?')} samples skipped")
        continue
    windows.append(frame)
    if len(windows) >= max_windows:
        break

print(f"Received {len(windows)} windows")
if windows:
    print(f"Columns: {windows[0].column_names}")
    meta = {k.decode(): v.decode() for k, v in (windows[0].schema.metadata or {}).items()}
    print(f"Metadata: {meta}")

In [ ]:
# Plot the latest spectrum
if windows:
    table = windows[-1]
    freqs = table.column('frequency_hz').to_numpy()
    mags = table.column('VA_mag_db').to_numpy()
    meta = {k.decode(): v.decode() for k, v in (table.schema.metadata or {}).items()}

    title = 'Live Spectral Snapshot (VA)'
    if 'thd_pct_VA' in meta:
        title += f" — THD {float(meta['thd_pct_VA']):.1f}%"

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.set_title(title)
    ax.plot(freqs, mags, 'k-', linewidth=0.5)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Magnitude (dB)')
    ax.set_xlim(0, 3000)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 3. Reconfigure the spectral stream

You can change channels, mode, FFT size, or frequency range on a live
connection by sending JSON control messages, without reconnecting:

- `{"type": "set_channels", "channels": ["VB", "IB"]}`
- `{"type": "set_mode", "mode": "fixed", "fft_size": 8192}`
- `{"type": "set_mode", "mode": "cycle_aligned", "cycles": 12}`
- `{"type": "set_freq_range", "freq_min": 0, "freq_max": 3000}`
- `{"type": "set_include_phase", "include_phase": true}`
- `{"type": "pause"}` / `{"type": "resume"}`

In [ ]:
# Connect, switch to VB in fixed mode with a larger FFT window, collect a few windows
ws_url = GATEWAY.replace('http://', 'ws://') + '/api/ws/spectral?channels=VA'
ws = websocket.create_connection(ws_url, timeout=10)

# Reconfigure via control messages (no reconnect needed)
ws.send(json.dumps({'type': 'set_channels', 'channels': ['VB']}))
ws.send(json.dumps({'type': 'set_mode', 'mode': 'fixed', 'fft_size': 8192}))
print("Sent reconfiguration: channels=[VB], mode=fixed, fft_size=8192")

# Collect a few windows with the new config (binary Arrow IPC frames)
reconfigured = []
while len(reconfigured) < 3:
    opcode, data = ws.recv_data()
    if opcode == websocket.ABNF.OPCODE_BINARY:
        reconfigured.append(pa.ipc.open_stream(data).read_all())

ws.close()
print(f"Received {len(reconfigured)} windows after reconfiguration")
if reconfigured:
    print(f"Columns: {reconfigured[-1].column_names}")

## 4. CPOW waveform streaming

The CPOW WebSocket at `/api/ws/cpow_stream` delivers raw waveform samples as
Arrow IPC binary messages. Each message contains ~512 rows (16 ms at 32 kHz).

Text messages are JSON gap markers (`{"type": "gap", "skipped_samples": N}`)
indicating that the reader fell behind the real-time stream.

In [ ]:
# Collect ~100 ms of live waveform data (about 6-7 messages)
batches = []
gaps = []
target_samples = 3200  # 100 ms at 32 kHz
total_samples = 0

print("Collecting live waveform data...")
for msg in connect_cpow_stream(gateway_url=GATEWAY):
    if isinstance(msg, dict):
        # Gap marker
        gaps.append(msg)
        print(f"  Gap: {msg.get('skipped_samples', '?')} samples skipped")
    else:
        # Arrow RecordBatch
        batches.append(msg)
        total_samples += len(msg)
        if total_samples >= target_samples:
            break

print(
    f"Collected {len(batches)} batches, {total_samples} total samples ({total_samples / 32000 * 1000:.0f} ms)"
)
if gaps:
    print(f"Encountered {len(gaps)} gap(s)")

In [ ]:
# Concatenate batches into a single table and plot
if batches:
    combined = pa.Table.from_batches(batches)
    df = combined.to_pandas()
    n = min(3200, len(df))
    time_ms = np.arange(n) / 32.0

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.set_title('Live Waveform Snapshot')
    for col, color in [('VA', 'black'), ('VB', 'red'), ('VC', 'blue')]:
        if col in df.columns:
            ax.plot(time_ms, df[col].values[:n], color=color, label=col, alpha=0.8, linewidth=0.5)
    ax.set_xlabel('Elapsed time (ms)')
    ax.set_ylabel('Voltage')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Snapshot capture

Capture a few seconds of live waveform data and save it to a local parquet file
for offline analysis.

> **Tip:** For longer captures or automated use, run `equser snapshot` from the command line.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

capture_duration_sec = 2
target_samples = capture_duration_sec * 32000

batches = []
total_samples = 0

print(f"Capturing {capture_duration_sec} seconds of live data...")
for msg in connect_cpow_stream(gateway_url=GATEWAY):
    if isinstance(msg, dict):
        continue  # Skip gap markers
    batches.append(msg)
    total_samples += len(msg)
    if total_samples >= target_samples:
        break

if batches:
    combined = pa.Table.from_batches(batches)
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    output_path = Path(f'snapshot_{timestamp}.parquet')
    pq.write_table(combined, output_path)
    print(f"Saved {len(combined)} samples to {output_path}")
    print(f"  Duration: {len(combined) / 32000:.2f} seconds")
    print(f"  Size: {output_path.stat().st_size / 1024:.0f} KB")
else:
    print("No data received.")

## Next steps

- **Tutorial 1** (`01-parquet-files.ipynb`): Work with parquet files directly for offline analysis.
- **Tutorial 2** (`02-local-duckdb.ipynb`): Run SQL queries directly against parquet files with DuckDB.
- **Tutorial 3** (`03-backend-api.ipynb`): Query historical data through the REST API.
- **Harmonic analysis** (`../analysis/harmonic-analysis.ipynb`): FFT-based analysis of CPOW data.
- **Snapshot tool**: Run `equser snapshot` from the command line for automated waveform capture.